In [65]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from reportlab.pdfgen import canvas
from reportlab.platypus import Table, TableStyle
from reportlab.lib import colors
from openpyxl import Workbook
from openpyxl.styles import (
    Font, PatternFill, Border, Side, Alignment
)
from openpyxl.utils import get_column_letter
%matplotlib inline

In [66]:
df_deudas_despues_julio = pd.read_csv(
    '../data/mora_despues_julio.csv', 
    sep=';', 
    decimal=',', 
    encoding='latin-1'
)

df_morosos_anteriores_julio = pd.read_csv(
    "../data/deudas_anteriores_julio.csv",
    sep=";",
    decimal=",",
    encoding="latin-1"
)

In [67]:
# Variables de entrada

pdf = canvas.Canvas("carta_mora16.pdf")
current_date = '15 de octubre de 2026'
fecha_corte = "12 de octubre 2026"
meses_de_mora = 1

In [68]:
df_deudas_despues_julio['jul-26'] = df_deudas_despues_julio['jul-26'].fillna('')
df_deudas_despues_julio['ago-26'] = df_deudas_despues_julio['ago-26'].fillna('')
df_deudas_despues_julio['sep-26'] = df_deudas_despues_julio['sep-26'].fillna('')
df_deudas_despues_julio['oct-26'] = df_deudas_despues_julio['oct-26'].fillna('')
df_deudas_despues_julio['nov-26'] = df_deudas_despues_julio['nov-26'].fillna('')
df_deudas_despues_julio['dic-26'] = df_deudas_despues_julio['dic-26'].fillna('')
df_deudas_despues_julio['ene-27'] = df_deudas_despues_julio['ene-27'].fillna('')
df_deudas_despues_julio['feb-27'] = df_deudas_despues_julio['feb-27'].fillna('')
df_deudas_despues_julio['mar-27'] = df_deudas_despues_julio['mar-27'].fillna('')
df_deudas_despues_julio['abr-27'] = df_deudas_despues_julio['abr-27'].fillna('')
df_deudas_despues_julio['may-27'] = df_deudas_despues_julio['may-27'].fillna('')
df_deudas_despues_julio['jun-27'] = df_deudas_despues_julio['jun-27'].fillna('')
df_deudas_despues_julio['PISO'] = df_deudas_despues_julio['PISO'].astype('Int64')
df_deudas_despues_julio['3y4 dormi'] = df_deudas_despues_julio['3y4 dormi'].fillna('')
df_deudas_despues_julio['Morosos'] = df_deudas_despues_julio['Morosos'].fillna('')

df_morosos_anteriores_julio['PISO'] = df_morosos_anteriores_julio['PISO'].astype('Int64')
df_mororsos_sin_duplicados = pd.DataFrame({
    'LETRA': df_morosos_anteriores_julio['LETRA'],
    'PISO': df_morosos_anteriores_julio['PISO']
})
df_mororsos_sin_duplicados = df_mororsos_sin_duplicados.drop_duplicates()
#df_morosos_anteriores_julio.info()

In [69]:
def generateTable(data):
    table = Table(data)
    # Definimos los estilos que queremos aplicar.
    # La lista de estilos contiene tuplas que especifican qué hacer.
    style = TableStyle([
        # Estilo para la fila de encabezados (fila 0, todas las columnas)
        ('BACKGROUND', (0, 0), (-1, 0), colors.grey),          # Fondo gris
        ('TEXTCOLOR', (0, 0), (-1, 0), colors.whitesmoke),     # Texto blanco
        ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),       # Fuente en negrita
        ('FONTSIZE', (0, 0), (-1, 0), 12),                     # Tamaño de fuente
        ('ALIGN', (0, 0), (-1, 0), 'CENTER'),                  # Alineación centrada
    
        # Estilo para el resto de las filas (de la 1 en adelante)
        ('BACKGROUND', (0, 1), (-1, -1), colors.beige),        # Fondo beige
        ('FONTNAME', (0, 1), (-1, -1), 'Helvetica'),
        ('FONTSIZE', (0, 1), (-1, -1), 10),
        ('ALIGN', (2, 1), (-1, -1), 'CENTER'),                 # Centrar columnas de montos
    
        # Estilo para toda la tabla
        ('GRID', (0, 0), (-1, -1), 1, colors.black),           # Poner una cuadrícula negra
        ('VALIGN', (0, 0), (-1, -1), 'MIDDLE'),                # Alineación vertical media
    ])
    # Aplicamos los estilos a la tabla.
    table.setStyle(style)
    
def formato_notificacion_mora(floor, 
                    number_of_department, 
                    user_name, 
                    current_date,
                    montos_x_mes,
                    total_adeudado,
                    fecha_corte,
                    montos_x_mes_fila1):
    building_name = "Edificio Miraflores Norte"
    city_date = "La Paz, "+current_date
    departatment = "Departamento "+str(floor)+str(number_of_department)
    font_bold_style = "Helvetica-Bold"
    font_normal_style = "Helvetica"
    font_normal_size = 11
    font_medium_size = 12
    font_big_size = 16
    interlineado = 580
    parrafo = [
        "Por medio de la presente, la Administración del "+building_name+" se dirige a usted",
        "para comunicarle que, de acuerdo con nuestros registros administrativos, el departamento",
        str(floor)+str(number_of_department)+" presenta un saldo pendiente correspondiente al pago de expensas comunes, con",
        "corte al "+fecha_corte
    ]
    parrafo2 = [
        "Por tal motivo, solicitamos regularizar el saldo pendiente a la brevedad posible, a fin de",
        "mantener al día las obligaciones correspondientes a su unidad y contribuir al adecuado",
        "funcionamiento y mantenimiento de las áreas y servicios comunes del edificio."
    ]
    parrafo3 = [
        "En caso de que el pago ya hubiera sido efectuado, agradeceremos presentar o enviar el",
        "correspondiente comprobante a la Administración para realizar la verificación y",
        "actualización de nuestros registros."
    ]
    
    # Título
    pdf.setFont(font_bold_style, font_big_size)
    pdf.drawCentredString(300, 780, building_name)
    
    # Subtítulo
    pdf.setFont(font_normal_style, font_normal_size)
    pdf.drawCentredString(300, 760, "ADMINISTRACIÓN")
    
    # Fecha
    pdf.drawString(370, 720, city_date)
    
    # Destinatario
    pdf.setFont(font_bold_style, font_normal_size)
    pdf.drawString(80, 680, "Señor(a):")
    
    pdf.setFont(font_normal_style, font_normal_size)
    pdf.drawString(80, 665, user_name)
    pdf.drawString(80, 650, departatment)
    pdf.drawString(80, 635, 'Presente.-')
    
    # Referencia
    pdf.setFont(font_bold_style, font_normal_size)
    pdf.drawString(80, 600, "REF.: NOTIFICACIÓN DE MORA POR EXPENSAS")
    
    # Texto
    pdf.setFont(font_normal_style, font_normal_size)

    for linea in parrafo:
        interlineado = interlineado - 15
        pdf.drawString(
            80,
            interlineado,
            linea
        )
    
    # Datos de la deuda
    interlineado = interlineado - 20
    pdf.setFont(font_bold_style, font_normal_size)
    pdf.drawString(80, interlineado, "DETALLE DE LA DEUDA")

    interlineado = interlineado - 20
    pdf.setFont(font_normal_style, font_normal_size)
    pdf.drawString(80, interlineado, "Períodos adeudados:")

    ancho_columnas = [113, 113, 113, 113] 
    
    #tabla = Table(montos_x_mes, colWidths=ancho_columnas)
    tabla = Table(montos_x_mes)
    tabla.setStyle(TableStyle([
        # dibuja una cuadricula desde la primera cuadricula arriba a la izquierda hasta la última fila y columna de color negro con grosor 1
        ("GRID", (0, 0), (-1, -1), 1, colors.black), 
        ("BACKGROUND", (0, 0), (-1, 0), colors.lightgrey),
        ("ALIGN", (1, 0), (-1, -1), "CENTER"),
        ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
        ("FONTNAME", (-1, 0), (-1, -1), "Helvetica-Bold"),
    ]))
    interlineado = interlineado - 50
    tabla.wrapOn(pdf, 500, 100)
    tabla.drawOn(pdf, 80, interlineado)

    if len(montos_x_mes_fila1) > 0:
        tabla1 = Table(montos_x_mes_fila1)
        tabla1.setStyle(TableStyle([
            # dibuja una cuadricula desde la primera cuadricula arriba a la izquierda hasta la última fila y columna de color negro con grosor 1
            ("GRID", (0, 0), (-1, -1), 1, colors.black), 
            ("BACKGROUND", (0, 0), (-1, 0), colors.lightgrey),
            ("ALIGN", (1, 0), (-1, -1), "CENTER"),
            ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
            ("FONTNAME", (-1, 0), (-1, -1), "Helvetica-Bold"),
        ]))
        interlineado = interlineado - 50
        tabla1.wrapOn(pdf, 500, 100)
        tabla1.drawOn(pdf, 80, interlineado)

    interlineado = interlineado - 20
    pdf.setFont(font_bold_style, font_medium_size)
    pdf.drawString(80, interlineado, f"TOTAL ADEUDADO: Bs. {total_adeudado}")

    interlineado = interlineado - 20
    pdf.setFont(font_normal_style, font_normal_size)
    for linea2 in parrafo2:
        pdf.drawString(
            80,
            interlineado,
            linea2
        )
        interlineado = interlineado - 15
    interlineado = interlineado - 5

    for linea3 in parrafo3:
        pdf.drawString(
            80,
            interlineado,
            linea3
        )
        interlineado = interlineado - 15

    # Firma
    interlineado = interlineado - 20
    pdf.setFont("Helvetica", 11)
    pdf.drawString(80, interlineado, "Atentamente,")
    interlineado = interlineado - 80
    pdf.drawString(80, interlineado, "____________________________")
    interlineado = interlineado - 15
    pdf.drawString(80, interlineado, "Administrador")
    interlineado = interlineado - 15
    pdf.drawString(80, interlineado, "Edif. " + building_name)
    pdf.showPage()

In [70]:
def verifica_deudas_anteriores_julio(LETRA, PISO):
    pass
def es_deudor_moroso(LETRA, PISO):
    if ((df_mororsos_sin_duplicados["LETRA"] == LETRA) & (df_mororsos_sin_duplicados["PISO"] == PISO)).any():
        return True
    else:
        return False

In [71]:
# Genera notifiacaciones
def genera_notificaciones_pdf():
    for indice, fila in df_deudas_despues_julio.iterrows():
        esDeudorAntesJulio = False
        esDeudorDespuesJulio = False
        monto_adeudado = 0
        montos_x_mes = [['Concepto', 'Expensas']]
        montos_x_mes_fila1 = []
        nro_columnas = 0
        # Busca mora anterior a Julio
        if es_deudor_moroso(fila["LETRA"], fila["PISO"]):
            for indicem, filam in df_morosos_anteriores_julio.iterrows():
                #print(filam["Mes "])
                if fila["LETRA"] == filam["LETRA"] and fila["PISO"] == filam["PISO"]:
                    if filam["Estado"] == "No pagado" or filam["Estado"] == "Pendiente":
                        esDeudorAntesJulio = True
                        monto_adeudado = monto_adeudado + float(filam["Saldo"])
                        monto_adeudado = round(monto_adeudado, 2)
                        nro_columnas = nro_columnas + 1
                        if nro_columnas < 11:
                            montos_x_mes.append([str(filam["Mes"]), filam["Saldo"]])
                        else:
                            montos_x_mes_fila1.append([str(filam["Mes"]), filam["Saldo"]])
        # Esta condicion es solo para deudores desde julio            
        #if str(fila["3y4 dormi"]).strip() != "x" and str(fila["Morosos"]).strip() == "x" and str(fila["escidal"]).strip() != "x":
        # Esta condicion es para deudores morosos antes de julio
        # if str(fila["Morosos"]).strip() == "x" and str(fila["escidal"]).strip() != "x":
        # Deudas que no correspondan a CIDAL
        if str(fila["escidal"]).strip() != "x":
            numero_meses_adeudados = 0
            for nombre_mes, monto in fila.iloc[3:14].items():
                if str(monto).strip() != "":
                    numero_meses_adeudados = numero_meses_adeudados + 1
                    monto_adeudado = round(monto_adeudado + monto, 2)
                    nro_columnas = nro_columnas + 1
                    if nro_columnas < 11:
                        montos_x_mes.append([nombre_mes, monto])
                    else:
                        montos_x_mes_fila1.append([nombre_mes, monto])
            
            if numero_meses_adeudados >= meses_de_mora:
                esDeudorDespuesJulio = True
                monto_adeudado = round(monto_adeudado, 2)
                #print(f"{str(indice)} El copropieatario {fila["LETRA"]}{fila["PISO"]} - {fila["NOMBRE Y APELLIDO"]} tiene {str(meses_de_mora)} meses de duda, meses adeudados {str(numero_meses_adeudados)}")
        if esDeudorAntesJulio or esDeudorDespuesJulio:
            nro_columnas = nro_columnas + 1
            if nro_columnas < 11:
                montos_x_mes.append(['Total', monto_adeudado])
            else:
                montos_x_mes_fila1.append(['Total', monto_adeudado])
            formato_notificacion_mora(fila["PISO"], fila["LETRA"], fila["NOMBRE Y APELLIDO"], current_date, [list(x) for x in zip(*montos_x_mes)], monto_adeudado, fecha_corte, [list(x) for x in zip(*montos_x_mes_fila1)])
        
    # Guardar
    pdf.save()

In [72]:
# Genera cuadro de mora consolidado
def genera_cuadro_mora():
    montos_x_mes = []
    for indice, fila in df_deudas_despues_julio.iterrows():
        esDeudorAntesJulio = False
        esDeudorDespuesJulio = False
        monto_adeudado = 0
        nro_columnas = 0
        datos_vecino = {}
        datos_vecino["nombre"] = fila["NOMBRE Y APELLIDO"]
        datos_vecino["departamento"] = f"{fila["PISO"]}{fila["LETRA"]}"
        datos_vecino["deuda"] = {}
        #montos_x_mes.append(datos_vecino)
        # Busca mora anterior a Julio
        if es_deudor_moroso(fila["LETRA"], fila["PISO"]):
            for indicem, filam in df_morosos_anteriores_julio.iterrows():
                if fila["LETRA"] == filam["LETRA"] and fila["PISO"] == filam["PISO"]:
                    if filam["Estado"] == "No pagado" or filam["Estado"] == "Pendiente":
                        esDeudorAntesJulio = True
                        monto_adeudado = monto_adeudado + float(filam["Saldo"])
                        monto_adeudado = round(monto_adeudado, 2)
                        datos_vecino["deuda"][str(filam["Mes"])] = filam["Saldo"]

        # Busca mora después de julio
        if str(fila["escidal"]).strip() != "x":
            numero_meses_adeudados = 0
            for nombre_mes, monto in fila.iloc[3:14].items():
                if str(monto).strip() != "":
                    numero_meses_adeudados = numero_meses_adeudados + 1
                    monto_adeudado = round(monto_adeudado + monto, 2)
                    datos_vecino["deuda"][nombre_mes] = monto
    
            if numero_meses_adeudados >= meses_de_mora:
                esDeudorDespuesJulio = True
                monto_adeudado = round(monto_adeudado, 2)
        if esDeudorAntesJulio or esDeudorDespuesJulio:
            montos_x_mes.append(datos_vecino)
    print(montos_x_mes)

    wb = Workbook()
    ws = wb.active
    ws.title = "Mora"
    
    # Colores
    color_titulo = "D9EAF7"
    color_encabezado = "B4C7E7"
    color_total = "E2F0D9"
    color_borde = "808080"
    
    # Bordes
    borde = Border(
        left=Side(style="thin", color=color_borde),
        right=Side(style="thin", color=color_borde),
        top=Side(style="thin", color=color_borde),
        bottom=Side(style="thin", color=color_borde)
    )
    
    # ==========================================
    # 3. RECORRER LOS VECINOS
    # ==========================================
    
    fila = 1
    
    for vecino in montos_x_mes:
    
        # Obtener los meses adeudados
        deuda = vecino["deuda"]
        meses = list(deuda.keys())
    
        # Crear DataFrame
        datos = {
            "Concepto": ["Expensas"],
            **{
                mes: [monto]
                for mes, monto in deuda.items()
            }
        }
    
        df = pd.DataFrame(datos)
    
        # Calcular total
        df["Total"] = df[meses].sum(axis=1)
    
        # --------------------------------------
        # TÍTULO DEL VECINO
        # --------------------------------------
    
        ws.cell(
            row=fila,
            column=1,
            value=(
                f"Departamento {vecino['departamento']} - "
                f"{vecino['nombre']}"
            )
        )
    
        # Dar formato al título
        ultima_columna = len(df.columns)
    
        ws.merge_cells(
            start_row=fila,
            start_column=1,
            end_row=fila,
            end_column=ultima_columna
        )
    
        celda = ws.cell(fila, 1)
        celda.font = Font(
            bold=True, size=12, color="17365D"
        )
        celda.fill = PatternFill(
            "solid", fgColor=color_titulo
        )
        celda.alignment = Alignment(
            horizontal="left", vertical="center"
        )
    
        ws.row_dimensions[fila].height = 22
    
        fila += 1
    
        # --------------------------------------
        # ENCABEZADOS DE LA TABLA
        # --------------------------------------
    
        fila_encabezado = fila
    
        for columna, nombre_columna in enumerate(
            df.columns, start=1
        ):
            celda = ws.cell(
                row=fila,
                column=columna,
                value=nombre_columna
            )
    
            celda.font = Font(bold=True)
            celda.fill = PatternFill(
                "solid", fgColor=color_encabezado
            )
            celda.alignment = Alignment(
                horizontal="center",
                vertical="center"
            )
            celda.border = borde
    
        fila += 1
    
        # --------------------------------------
        # DATOS DE LA TABLA
        # --------------------------------------
    
        for columna, valor in enumerate(
            df.iloc[0], start=1
        ):
            celda = ws.cell(
                row=fila,
                column=columna,
                value=valor
            )
    
            celda.border = borde
    
            if columna == 1:
                celda.alignment = Alignment(
                    horizontal="left"
                )
            else:
                celda.alignment = Alignment(
                    horizontal="right"
                )
                celda.number_format = '#,##0.00'
    
        # Resaltar la celda del total
        celda_total = ws.cell(
            row=fila,
            column=ultima_columna
        )
    
        celda_total.font = Font(bold=True)
        celda_total.fill = PatternFill(
            "solid", fgColor=color_total
        )
    
        # Siguiente vecino: dejar una fila vacía
        fila += 2
    
    # ==========================================
    # 4. AJUSTAR ANCHO DE COLUMNAS
    # ==========================================
    
    for columna in range(1, ws.max_column + 1):
    
        letra = get_column_letter(columna)
        ancho_maximo = 0
    
        for celda in ws[letra]:
            if celda.value is not None:
                ancho_maximo = max(
                    ancho_maximo,
                    len(str(celda.value))
                )
    
        ws.column_dimensions[letra].width = min(
            max(ancho_maximo + 3, 12), 35
        )
    
    # ==========================================
    # 5. CONFIGURAR IMPRESIÓN
    # ==========================================
    
    ws.sheet_properties.pageSetUpPr.fitToPage = True
    ws.page_setup.fitToWidth = 1
    ws.page_setup.fitToHeight = 0
    ws.page_setup.orientation = "landscape"
    ws.page_setup.paperSize = ws.PAPERSIZE_A4
    
    ws.page_margins.left = 0.3
    ws.page_margins.right = 0.3
    ws.page_margins.top = 0.5
    ws.page_margins.bottom = 0.5
    
    # ==========================================
    # 6. GUARDAR EL ARCHIVO
    # ==========================================
    
    wb.save("reporte_mora_miraflores.xlsx")

        

In [73]:
if __name__ == "__main__":
    #genera_notificaciones_pdf()
    genera_cuadro_mora()
    print("Generación correcta de mora")

[{'nombre': 'JOSE MANUEL BUSTILLOS', 'departamento': '3C', 'deuda': {'jul-26': 157.48, 'ago-26': 163.53, 'sep-26': 163.02}}, {'nombre': 'JULIETA MAMANI DE NINA', 'departamento': '3E', 'deuda': {'jun-25': 209.0, 'jul-25': 217.0, 'ago-25': 220.05, 'sep-25': 201.11, 'oct-25': 220.27, 'nov-25': 219.73, 'dic-25': 212.42, 'ene-26': 217.0, 'feb-26': 258.4, 'mar-26': 258.02, 'abr-26': 271.94, 'may-26': 255.3, 'jun-26': 280.97, 'jul-26': 268.06, 'ago-26': 238.98, 'sep-26': 237.66}}, {'nombre': 'MANUEL A. LIRA ORTIZ', 'departamento': '4B', 'deuda': {'feb-26': 194.31, 'mar-26': 162.49, 'abr-26': 167.19, 'may-26': 167.19, 'jun-26': 191.31, 'jul-26': 196.17, 'ago-26': 190.87, 'sep-26': 183.74}}, {'nombre': 'SIRLEY VALENCIA/ANGEL VALENCVIA SIRPA', 'departamento': '4C', 'deuda': {'sep-26': 173.45}}, {'nombre': 'ELIZABETH PAREDES SARDON', 'departamento': '4E', 'deuda': {'sep-26': 342.24}}, {'nombre': 'LOURDES LIMA TELLEZ', 'departamento': '4F', 'deuda': {'sep-26': 185.45}}, {'nombre': 'MAIDA MENDIA/ J